# LAS CMS — Phase 3.1: PII Removal
---

In [1]:
import pandas as pd, numpy as np, hashlib, re, json
from pathlib import Path
from datetime import datetime

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs/anonymized")
MAPPING_DIR = Path("../outputs/pii_mapping")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MAPPING_DIR.mkdir(parents=True, exist_ok=True)
HASH_SALT = "LAS_CMS_2026_CONFIDENTIAL"
print("✓ Setup")

✓ Setup


In [2]:
TABLE_COLUMNS = {
    "programs": ["id","programName","caseReferred","caseReferredsub","caseReferedsubtext","districtName","zakatEligible","interviewDate","interviewerName","clientName","partyName","complainantName","relationShip","fatherHusbandName","contactNumber","alternativecontactNumber","cnic","gender","age","religion","religionOther","caseFacts","caseSubmittedFAppro","caseApprovalStatus","approvalDate","case_not_filed","case_not_filed_reason","vakalatnamaSubmissionDate","caseFileDate","lawyer1","lawyer2","reasonOfChange","courtName","levelOfCourt","caseNumber","firNumber","policeStation","natureOfCase","typeOfCase","mainCaseCategory","sgbvSubCat","caseFiledUnderAct","caseFiledOther","nextHearing","currentCaseStatus","caseDecision","caseDisposalDate","ctcStatus","caseStage","caseStageother","additionalComment","created_at","updated_at","username","edited_by","primarykey","rejectionReason","UniqueNumber","uniqueYear","UniqueNumber2"],
    "hearings": ["id","programsID","caseNumber","primarykey","date","nextHearing","hearingUpdate","updated_at","created_at"],
    "programs_detail": ["id","programsid","programName","caseReferred","caseReferredsub","caseReferedsubtext","districtName","zakatEligible","interviewDate","interviewerName","clientName","complainantName","relationShip","fatherHusbandName","contactNumber","alternativecontactNumber","cnic","gender","age","religion","religionOther","caseFacts","caseSubmittedFAppro","caseApprovalStatus","approvalDate","case_not_filed","case_not_filed_reason","vakalatnamaSubmissionDate","caseFileDate","lawyer1","lawyer2","reasonOfChange","courtName","levelOfCourt","caseNumber","firNumber","policeStation","natureOfCase","typeOfCase","mainCaseCategory","sgbvSubCat","caseFiledUnderAct","caseFiledOther","nextHearing","currentCaseStatus","caseDecision","caseDisposalDate","ctcStatus","caseStage","caseStageother","additionalComment","created_at","updated_at","username","edited_by","primarykey","rejectionReason","UniqueNumber","uniqueYear","UniqueNumber2"],
    "users": ["id","google_id","name","email","google_email","google_avatar","provider","provider_token","provider_refresh_token","provider_token_expires_at","email_verified_at","password","remember_token","created_at","updated_at","created_by","allowed_programs","user_type","roles","last_login_at","last_login_ip"],
    "case_documents": ["id","program_id","uploaded_by","file_name","file_path","created_at","updated_at"],
    "court": ["id","name","court_long","district_id","districtName","address","is_active","created_by","created_on","last_modified_by","last_modified_on","is_deleted","deleted_by","deleted_on","last_status_modified_on"],
    "interviews": ["id","created_by","status","created_at","updated_at"],
}

In [3]:
def load_table(name, data_dir):
    """Load table from TSV/CSV and assign correct column names."""
    from pathlib import Path
    
    schema_cols = TABLE_COLUMNS.get(name)
    df = pd.DataFrame()
    source = ""
    
    for ext, sep in [('.tsv', '\t'), ('.csv', '\t'), ('.csv', ',')]:
        fpath = data_dir / f"{name}{ext}"
        if not fpath.exists():
            continue
        try:
            # Read without header first to check
            test = pd.read_csv(fpath, sep=sep, nrows=2, header=None, on_bad_lines='skip')
            if test.empty or len(test.columns) < 3:
                continue
            
            # Read full file without header
            df = pd.read_csv(fpath, sep=sep, header=None, on_bad_lines='warn',
                            low_memory=False, na_values=['NULL','null','None',''])
            
            # Check if first row looks like a header (matches schema column names)
            first_row = [str(v).strip() for v in df.iloc[0].values[:5]]
            if schema_cols and first_row[0] == schema_cols[0]:
                # File HAS a header row — drop it, it's the column names
                df = df.iloc[1:].reset_index(drop=True)
            
            # Assign column names from schema
            if schema_cols:
                if len(df.columns) == len(schema_cols):
                    df.columns = schema_cols
                elif len(df.columns) > len(schema_cols):
                    df.columns = schema_cols + [f'extra_{i}' for i in range(len(df.columns) - len(schema_cols))]
                else:
                    df.columns = schema_cols[:len(df.columns)]
            
            source = f"{ext[1:]}→{len(df.columns)}cols"
            break
        except Exception as e:
            continue
    
    if df.empty:
        print(f"  ⚠ {name} — not found or empty")
        return pd.DataFrame()
    
    # Clean special values
    df = df.replace(['NULL','null','None','0000-00-00','0000-00-00 00:00:00'], pd.NA)
    
    print(f"  ✓ {name:30s} {len(df):>6,} rows × {len(df.columns):>2} cols  [{source}]")
    return df

In [4]:
programs        = load_table("programs", DATA_DIR)
hearings        = load_table("hearings", DATA_DIR)
programs_detail = load_table("programs_detail", DATA_DIR)
users           = load_table("users", DATA_DIR)

  ✓ programs                        3,886 rows × 60 cols  [tsv→60cols]


C:\Users\awais\AppData\Local\Temp\ipykernel_16460\1980960580.py:20: ParserWarning: Skipping line 426: expected 60 fields, saw 62
Skipping line 568: expected 60 fields, saw 61
Skipping line 1876: expected 60 fields, saw 61
Skipping line 2395: expected 60 fields, saw 61
Skipping line 2563: expected 60 fields, saw 61
Skipping line 2631: expected 60 fields, saw 61

  df = pd.read_csv(fpath, sep=sep, header=None, on_bad_lines='warn',
C:\Users\awais\AppData\Local\Temp\ipykernel_16460\1980960580.py:20: ParserWarning: Skipping line 304: expected 9 fields, saw 10
Skipping line 335: expected 9 fields, saw 10
Skipping line 420: expected 9 fields, saw 10
Skipping line 495: expected 9 fields, saw 10
Skipping line 497: expected 9 fields, saw 10
Skipping line 1056: expected 9 fields, saw 10
Skipping line 1660: expected 9 fields, saw 10
Skipping line 2630: expected 9 fields, saw 10
Skipping line 3574: expected 9 fields, saw 10
Skipping line 3689: expected 9 fields, saw 10
Skipping line 3787: expected 

  ✓ hearings                       13,740 rows ×  9 cols  [tsv→9cols]
  ✓ programs_detail                 6,362 rows × 60 cols  [tsv→60cols]
  ✓ users                              89 rows × 21 cols  [tsv→21cols]


C:\Users\awais\AppData\Local\Temp\ipykernel_16460\1980960580.py:20: ParserWarning: Skipping line 429: expected 60 fields, saw 61
Skipping line 571: expected 60 fields, saw 61
Skipping line 1880: expected 60 fields, saw 61
Skipping line 2401: expected 60 fields, saw 61
Skipping line 2569: expected 60 fields, saw 61
Skipping line 2637: expected 60 fields, saw 61
Skipping line 3463: expected 60 fields, saw 61
Skipping line 4710: expected 60 fields, saw 61

  df = pd.read_csv(fpath, sep=sep, header=None, on_bad_lines='warn',


## Anonymize

In [5]:
def hv(v, pfx=""):
    if pd.isna(v) or str(v).strip()=='': return v
    return f"{pfx}{hashlib.sha256(f'{HASH_SALT}:{str(v).strip()}'.encode()).hexdigest()[:8].upper()}"

def anon_cnic(c):
    if pd.isna(c) or str(c).strip()=='': return c
    return 'FAKE_PLACEHOLDER' if '00000-0000000-0' in str(c) else hv(c,"CNIC_")
def anon_name(n):
    return hv(str(n).strip(),"NAME_") if pd.notna(n) and str(n).strip()!='' else n
def anon_phone(p):
    return hv(str(p).strip(),"PHONE_") if pd.notna(p) and str(p).strip()!='' else p

def anon_text(text, names=None):
    if pd.isna(text) or str(text).strip()=='': return text
    t = str(text)
    t = re.sub(r'\d{5}-\d{7}-\d{1}','[CNIC_REDACTED]',t)
    t = re.sub(r'(\+92|0)3\d{2}[\s-]?\d{7}','[PHONE_REDACTED]',t)
    t = re.sub(r'\b\d{7,13}\b','[NUM_REDACTED]',t)
    if names:
        for n in names:
            if n and len(str(n))>2:
                t = re.sub(re.escape(str(n).strip()),'[NAME_REDACTED]',t,flags=re.IGNORECASE)
    return t
print("✓ Functions ready")

✓ Functions ready


In [6]:
# Collect names for text redaction
all_names = set()
for col in ['clientName','fatherHusbandName','complainantName','partyName','interviewerName']:
    if col in programs.columns:
        for n in programs[col].dropna().unique():
            ns = str(n).strip()
            if len(ns)>2: all_names.add(ns)
            for part in ns.split():
                if len(part)>2: all_names.add(part)
print(f"✓ {len(all_names)} names collected")

# PII mapping
mapping = []
for col,typ,fn in [('cnic','CNIC',anon_cnic),('clientName','NAME',anon_name),
                    ('fatherHusbandName','FATHER',anon_name),('contactNumber','PHONE',anon_phone)]:
    if col in programs.columns:
        for v in programs[col].dropna().unique():
            mapping.append({'type':typ,'original':str(v),'anonymized':fn(v)})
pd.DataFrame(mapping).to_csv(MAPPING_DIR/"pii_mapping_CONFIDENTIAL.csv", index=False)
print(f"✓ Mapping: {len(mapping):,} entries (CONFIDENTIAL!)")

✓ 6570 names collected
✓ Mapping: 7,370 entries (CONFIDENTIAL!)


In [7]:
# Anonymize structured fields
pa = programs.copy()
for f,lbl,fn in [('clientName','Client',anon_name),('fatherHusbandName','Father',anon_name),
                  ('complainantName','Complainant',anon_name),('partyName','Party',anon_name),
                  ('interviewerName','Interviewer',anon_name),
                  ('cnic','CNIC',anon_cnic),('contactNumber','Phone',anon_phone),
                  ('alternativecontactNumber','Alt Phone',anon_phone)]:
    if f in pa.columns:
        pa[f] = pa[f].apply(fn); print(f"✓ {lbl} anonymized")

# Free text
names_list = list(all_names)
if 'caseFacts' in pa.columns:
    print("Anonymizing caseFacts..."); pa['caseFacts'] = pa['caseFacts'].apply(lambda x: anon_text(x,names_list)); print("✓ Done")
if 'additionalComment' in pa.columns:
    pa['additionalComment'] = pa['additionalComment'].apply(lambda x: anon_text(x,names_list))

ha = hearings.copy() if not hearings.empty else pd.DataFrame()
if not ha.empty and 'hearingUpdate' in ha.columns:
    print("Anonymizing hearingUpdate..."); ha['hearingUpdate'] = ha['hearingUpdate'].apply(lambda x: anon_text(x,names_list)); print("✓ Done")

# Save
pa.to_csv(OUTPUT_DIR/"programs_anonymized.csv", index=False); print(f"✓ programs ({len(pa):,})")
if not ha.empty: ha.to_csv(OUTPUT_DIR/"hearings_anonymized.csv", index=False); print(f"✓ hearings ({len(ha):,})")
if not programs_detail.empty:
    pda = programs_detail.copy()
    for f,_,fn in [('clientName','',anon_name),('cnic','',anon_cnic),('contactNumber','',anon_phone),('fatherHusbandName','',anon_name)]:
        if f in pda.columns: pda[f] = pda[f].apply(fn)
    pda.to_csv(OUTPUT_DIR/"programs_detail_anonymized.csv", index=False); print(f"✓ programs_detail ({len(pda):,})")

print("\n✅ PII removal complete! Next: 03_data_cleaning.ipynb")

✓ Client anonymized
✓ Father anonymized
✓ Complainant anonymized
✓ Party anonymized
✓ Interviewer anonymized
✓ CNIC anonymized
✓ Phone anonymized
✓ Alt Phone anonymized
Anonymizing caseFacts...
✓ Done
Anonymizing hearingUpdate...
✓ Done
✓ programs (3,886)
✓ hearings (13,740)
✓ programs_detail (6,362)

✅ PII removal complete! Next: 03_data_cleaning.ipynb
